In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os
load_dotenv()
google_api_key = os.getenv('GOOGLE_API_KEY')
model = "gemini-2.0-flash"
chat = ChatGoogleGenerativeAI(
    model = model, temperature = 0.0,
)

In [2]:
from langchain.output_parsers import StructuredOutputParser, ResponseSchema

response_schema = [
    ResponseSchema(name = 'sentiment', description="Sentiment of the review: Positive, Negative or Neutral"),
    ResponseSchema(name = 'highlights', description="Complains or Compliments in the review"),
    ResponseSchema(name = 'summary', description="Summary of the review"),
]

output_parser = StructuredOutputParser.from_response_schemas(response_schema)
print(output_parser)


response_schemas=[ResponseSchema(name='sentiment', description='Sentiment of the review: Positive, Negative or Neutral', type='string'), ResponseSchema(name='highlights', description='Complains or Compliments in the review', type='string'), ResponseSchema(name='summary', description='Summary of the review', type='string')]


In [12]:
from langchain.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """
Extract out the following information from the ``{review}``:
{information}
"""
)

user_reviews = [
    "I bought this wireless mouse for my laptop and I have to say, it's exceeded my expectations. The connection is super fast with no lag at all, and the ergonomic design makes it really comfortable to use even for long hours of work. The battery life is outstanding — I’ve been using it for over a month and haven’t had to recharge it once. I also appreciate that it came with a USB-C charger instead of a micro USB. Overall, this mouse is a great value for the price and I’d highly recommend it to anyone looking for a reliable and affordable option.",
    
    "Unfortunately, the blender I purchased turned out to be a complete disappointment. Within two weeks, it started making a loud grinding noise whenever I tried to blend even soft fruits like bananas. Shortly after that, it just stopped turning on. The build quality feels cheap, and the motor seems underpowered for even basic kitchen tasks. I reached out to customer support but haven’t received a response yet. It’s frustrating because I had high hopes based on the reviews. I wouldn’t recommend this product unless you want to deal with returns and wasted time.",
    
    "This phone case is definitely stylish — I like the texture and the color choices. However, its protective quality is questionable. My phone slipped off the couch and fell barely two feet, yet the screen cracked in the corner. The edges of the case aren’t raised enough to protect the screen, and the corners don’t absorb shock well. If you just want your phone to look good, this might work. But if you're clumsy or want real protection, you might want to look elsewhere.",
    
    "I’ve been using this standing desk for over three months now, and it has genuinely improved my daily routine. I work from home full-time, and this desk has made a huge difference in terms of comfort and productivity. The motor is quiet and smooth when adjusting the height, and the surface is spacious enough to fit two monitors, a laptop, and some accessories. Assembly was easy too. It’s well worth the investment if you spend long hours at your desk and want a healthier way to work.",
    
    "I bought these earbuds mainly for workouts, and while they’re not perfect, they do the job well enough. The sound quality is decent with clear highs and a bit of bass, though nothing extraordinary. The main downside is the battery — I usually have to charge them every two days even with moderate use. The fit is snug, and they don’t fall out during runs, which is a big plus. If battery life isn’t a big concern, they’re a solid budget option.",
    
    "My experience with this product was extremely disappointing from the start. The packaging was flimsy and arrived torn. When I opened the box, the product inside had clearly been used or returned before — there were visible scratches and even some residue. On top of that, it didn’t function properly and gave an error message on the first use. I contacted support and was told to wait 5–7 business days for a resolution, which is absurd for a faulty product. Not recommended at all.",
    
    "I’m thrilled with these running shoes. As someone who trains for marathons, finding the right balance between cushioning and responsiveness is essential, and these shoes nailed it. They’re incredibly lightweight yet supportive, and the breathable mesh keeps my feet cool even during long runs. I've already logged over 100 kilometers in them, and they still feel as good as new. Definitely worth every penny for serious runners or anyone who’s on their feet a lot.",
    
    "The smartwatch is decent overall, especially for the price point. It tracks steps, heart rate, and even sleep pretty accurately. The problem is syncing — it took me over an hour to get it to connect with my Android phone, and the app isn’t very intuitive. Once set up, though, the notifications and features work fine. The screen is bright, and battery life lasts about 3 days on moderate use. It’s a good entry-level watch if you don’t mind a slightly frustrating setup process.",
    
    "I’ve been using these wireless headphones for over a month now, and I must say the sound quality is absolutely stunning. The bass is deep, vocals are crisp, and noise cancellation works like a charm even in crowded areas. The battery life easily lasts for 30+ hours on a single charge. The only downside is that the ear cushions feel slightly tight after long periods of use, but overall, it’s a fantastic buy for the price.",

    "The electric toothbrush is very effective at cleaning teeth thoroughly. My dentist even noticed the difference. It comes with multiple modes for sensitive teeth, whitening, and deep clean. I also like the sleek design and the smart timer feature. However, the charging cable is proprietary and short, which makes it less convenient to travel with.",

    "I got this smartwatch primarily for fitness tracking, and it delivers on most fronts. The step counter, heart rate monitor, and sleep tracking seem accurate. The screen is bright and responsive, and I love how seamlessly it integrates with my phone. My only complaint is that the battery drains faster when using GPS features, but it still lasts a full day.",

    "This blender is a powerhouse in the kitchen. It effortlessly crushes ice, blends smoothies, and even makes hot soup. The motor is powerful, and the blade design is excellent. It’s a bit noisy, but that’s expected given the performance. Cleaning it is relatively easy, especially with the self-clean mode. I’ve started using it daily for everything from breakfast to dinner prep.",

    "This laptop exceeded my expectations. The boot time is incredibly fast, thanks to the SSD. The display is vibrant with excellent resolution, making it perfect for both work and watching movies. I use it for programming and occasional gaming, and it handles both smoothly. The only issue is that it heats up slightly when multitasking heavily, so I had to buy a cooling pad.",

    "I’ve become a home barista thanks to this espresso machine. It heats up quickly and pulls rich, creamy shots of espresso. The built-in grinder is a big plus, and the steam wand creates smooth microfoam for lattes. Setup was a bit complex at first, and cleaning requires some effort, but it’s totally worth it for the quality of coffee I now get at home.",

    "I work from home and sit for long hours, and this chair has really improved my comfort. The lumbar support is great, and I can adjust the height, tilt, and armrests to fit my posture. The mesh back keeps me cool during hot days. One downside is that the wheels don’t roll smoothly on carpet, but otherwise, it's a solid ergonomic choice.",

    "This power bank is a lifesaver during travel. It charges my phone almost four times fully and still has juice left. The fast charging support is real — I get 50% charge in about 20 minutes. It’s a bit heavy to carry in a pocket but fits nicely in a bag. The LED indicators for battery percentage are a nice touch.",

    "I love this water bottle for daily use. It’s made from stainless steel and keeps water cold for over 24 hours, even when left in a hot car. The wide mouth makes it easy to clean and add ice cubes. It’s leak-proof and has a durable powder-coated finish that hasn’t chipped even after months of use. It’s slightly bulky, but the handle helps with portability.",

    "This robot vacuum has truly made my life easier. I just schedule it via the app, and it does a great job navigating around furniture. It transitions well between hardwood and rugs and even returns to its dock when it needs a charge. Sometimes it misses small corners, but for daily light cleaning, it’s very reliable. The dustbin is small, so I empty it every couple of days.",

]

formatted_prompt = prompt.format_prompt(
    review=user_reviews[10],
    information = output_parser.get_format_instructions()
)
print(formatted_prompt)

text='\nExtract out the following information from the ``I got this smartwatch primarily for fitness tracking, and it delivers on most fronts. The step counter, heart rate monitor, and sleep tracking seem accurate. The screen is bright and responsive, and I love how seamlessly it integrates with my phone. My only complaint is that the battery drains faster when using GPS features, but it still lasts a full day.``:\nThe output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":\n\n```json\n{\n\t"sentiment": string  // Sentiment of the review: Positive, Negative or Neutral\n\t"highlights": string  // Complains or Compliments in the review\n\t"summary": string  // Summary of the review\n}\n```\n'


In [13]:
response = chat.invoke(formatted_prompt)
structured_output = output_parser.parse(response.content)
print(structured_output)

{'sentiment': 'Positive', 'highlights': 'Accurate step counter, heart rate monitor, and sleep tracking. Bright and responsive screen. Seamless integration with phone. Battery drains faster when using GPS.', 'summary': 'The smartwatch is good for fitness tracking with accurate sensors and a bright screen. It integrates well with the phone, but the battery drains quickly with GPS use. Overall, the reviewer is satisfied.'}
